# Week 3 Session 1: Titanic with pandas: Read, Inspect, Clean
**Forge Academy | Data Science Foundations**
*Wraps up Week 2 Session 2 (working with real CSV data) and covers Week 3 Session 1 (Data Cleaning Lab).*

Today we take a real dataset and do what data people do every day: read it, look at it carefully, find the problems, fix them, and explain every fix. Then we ask it our first real questions.

**By the end of this session you can:**
1. Load a CSV into pandas, including only the parts you need.
2. Select rows and columns with `[]`, `loc`, `iloc` and filters.
3. Inspect and validate a table: types, unique values, missing values, duplicates, odd values.
4. Clean data and write a justification for every fix.
5. Create new columns, save a clean file, and answer first questions with `groupby` and `sort_values`.

**Setup:** keep `Titanic-Dataset.csv` in the **same folder** as this notebook. Open the notebook in VS Code, select your Python kernel, and run cells from top to bottom. Cells marked **Your turn** are for you to write code.


## 0. Recap: Module 3 in plain words

- **A CSV is just a table saved as text.** Each line is a row, commas separate the columns, and the first line is usually the column names.
- **Module 3 showed the long way:** opening files and reading them line by line. That exists and is good to know about, but it is slow to write and easy to get wrong.
- **In real work, everyone uses pandas.** pandas reads a whole CSV into a table called a **DataFrame** in one line, and gives us tools to inspect and clean it.
- So today we skip manual file reading and use the tool you will actually use at work.

| Word | Meaning |
|---|---|
| DataFrame | A table in pandas: rows and columns |
| Series | One column of a DataFrame |
| index | The row labels (0, 1, 2, ... by default) |
| `NaN` | A missing value ("Not a Number") |
| dtype | The type of a column: `int64` (whole numbers), `float64` (decimals), `object`/`str` (text), `bool` (True/False) |

**The dataset:** 891 Titanic passengers. `Survived` (1 = yes, 0 = no), `Pclass` (ticket class 1, 2, 3), `Name`, `Sex`, `Age`, `SibSp` (siblings and spouses aboard), `Parch` (parents and children aboard), `Ticket`, `Fare`, `Cabin`, `Embarked` (port: C = Cherbourg, Q = Queenstown, S = Southampton).

## 1. Reading data

We import pandas with the short name `pd`, which everyone uses.

In [ ]:
import pandas as pd

In [ ]:
raw = pd.read_csv("Titanic-Dataset.csv")   # one line: the whole file becomes a table
raw.head()

If you get `FileNotFoundError`, the CSV is not in the same folder as this notebook. Move it there and run the cell again.

`head()` shows the first 5 rows; `head(10)` shows 10. `tail()` shows the last rows.

In [ ]:
raw.tail(3)

### Useful `read_csv` options
You rarely need these, but they save time on big files.
- `nrows=` reads only the first few rows (a quick peek at a huge file).
- `usecols=` reads only the columns you need.
- `index_col=` uses a column as the row labels.

In [ ]:
peek = pd.read_csv("Titanic-Dataset.csv", nrows=5)
print(peek.shape)

In [ ]:
small = pd.read_csv("Titanic-Dataset.csv", usecols=["Name", "Sex", "Age", "Survived"])
small.head()

In [ ]:
by_id = pd.read_csv("Titanic-Dataset.csv", index_col="PassengerId")
by_id.head(3)

Notice that with `index_col`, `PassengerId` became the row labels on the left instead of a normal column.

### Your turn 1
Read only the `Name`, `Pclass` and `Fare` columns, for the first 8 rows, into a variable called `fares`. Show it.

In [ ]:
# Your code here


## 2. Selecting rows and columns

Before we can inspect or clean, we need to be able to point at exactly the data we want.

**One column** with single brackets gives a Series. **Several columns** need double brackets (a list inside), and give a DataFrame.

In [ ]:
raw["Age"].head()

In [ ]:
raw[["Name", "Age", "Survived"]].head()

**`iloc` selects by position** (row number, column number), like list indexing. **`loc` selects by label** (index label, column name).

In [ ]:
print(raw.iloc[0])          # first row
raw.iloc[0:3, 0:4]          # rows 0-2, first 4 columns

In [ ]:
raw.loc[0:3, ["Name", "Sex", "Age"]]   # with loc the end label IS included

**Boolean filtering** is the most useful one. A condition gives True/False for every row, and we keep the True rows.
Combine conditions with `&` (and) and `|` (or), and wrap each one in brackets.

In [ ]:
print((raw["Age"] < 10).head())   # True / False for each row
children = raw[raw["Age"] < 10]
print("Children under 10:", len(children))

In [ ]:
first_class_women = raw[(raw["Pclass"] == 1) & (raw["Sex"] == "female")]
print("First class women:", len(first_class_women))
first_class_women[["Name", "Age", "Fare"]].head()

In [ ]:
# loc can filter rows and pick columns at the same time
raw.loc[raw["Fare"] > 200, ["Name", "Pclass", "Fare"]]

### Your turn 2
1. Show the `Name` and `Age` of passengers older than 60.
2. How many third class passengers survived? (filter on two conditions, then use `len`)

In [ ]:
# Your code here


In [ ]:
# Your code here


## 3. Inspect and validate: find the problems before fixing anything

Good cleaning starts with good looking. We ask:
1. How big is it, and what types are the columns?
2. What values does each column actually contain?
3. What is missing?
4. Are there duplicates?
5. Are there values that make no sense?

In [ ]:
print("Rows and columns:", raw.shape)
print(list(raw.columns))

In [ ]:
raw.info()

Read the **Non-Null Count**: anything below 891 has gaps. Read the **Dtype**: does each type make sense? `Age` is a decimal (`float64`), even though ages are usually whole years. Keep that in mind.

In [ ]:
raw.dtypes

**Summary numbers.** `describe()` covers number columns. Scan min and max for anything strange.

In [ ]:
raw.describe()

`describe(include="object")` (or `include="all"`) summarises the text columns: how many unique values, the most common one (`top`) and how often it appears (`freq`).

In [ ]:
raw.describe(include="object")

**What values are there?** `unique()` lists distinct values, `nunique()` counts them, `value_counts()` counts each one. Use `dropna=False` so missing values show up.

In [ ]:
print(raw["Embarked"].unique())
print(raw["Sex"].nunique(), "distinct values in Sex")
raw.nunique()

In [ ]:
print(raw["Sex"].value_counts(dropna=False))
print()
print(raw["Embarked"].value_counts(dropna=False))

**Missing values.** `isnull().sum()` counts missing values per column. `isnull().mean()` gives the share, which is easier to judge.

In [ ]:
raw.isnull().sum()

In [ ]:
print(round(raw.isnull().mean() * 100, 1))

**Duplicates.** `duplicated()` marks rows that are exact copies of an earlier row.

In [ ]:
print("Duplicate rows:", raw.duplicated().sum())
print("Duplicate names:", raw["Name"].duplicated().sum())

**Odd and invalid values.** `describe()` showed a minimum Fare of 0, and the minimum Age is below 1. Let us look closer.

In [ ]:
free = raw[raw["Fare"] == 0]
print("Passengers with Fare 0:", len(free))
free[["Name", "Pclass", "Fare", "Ticket"]].head()

In [ ]:
# Ages that are not whole numbers
decimal_ages = raw[(raw["Age"] % 1) != 0]
print("Non-whole ages:", len(decimal_ages))
decimal_ages[["Name", "Age"]].head(8)

Ages under 1 are babies (0.42 is about 5 months), so those are real. Ages like 28.5 are estimates in the original records. These are not errors, but we should know about them.

In [ ]:
# The highest fares: real or a typo?
raw.sort_values("Fare", ascending=False)[["Name", "Pclass", "Fare"]].head()

### Your turn 3
1. How many distinct ticket numbers are there? Compare with the number of rows. What might that mean?
2. Show `value_counts()` for `Pclass` and for `Parch`.

In [ ]:
# Your code here


In [ ]:
# Your code here


### Your turn 4: write your problem list
In this cell, list **at least 4 problems** you found and why each could matter for analysis.

1.
2.
3.
4.

## 4. Cleaning lab: fix and justify

**Rule 1: never change the raw data.** Work on a copy, so you can always compare and go back.

**Rule 2: every fix needs a justification.** One line: what you did, why, and any limitation. A fix without a reason is a guess.

**Rule 3: check the result.** Print counts before and after so nothing changes silently.

In [ ]:
df = raw.copy()
print("Raw:", raw.shape, " Working copy:", df.shape)

### Fix 1: Missing Embarked (2 rows)
Embarked is a category, so we fill with the **mode** (most common value).

In [ ]:
print(df[df["Embarked"].isnull()][["Name", "Fare", "Pclass"]])
port_mode = df["Embarked"].mode()[0]
print("Most common port:", port_mode)
df["Embarked"] = df["Embarked"].fillna(port_mode)
print("Missing Embarked after fix:", df["Embarked"].isnull().sum())

**Justification:** Only 2 rows, and a median makes no sense for a category. The most common port changes the overall picture very little.

### Fix 2: Missing Age (177 rows, about 20%)
**Option A: one median for everyone.** Simple and common.

In [ ]:
print("Median age:", df["Age"].median(), " Mean age:", round(df["Age"].mean(), 1))

The median is not pulled around by a few very old or very young passengers, so it is a safer "typical" value than the mean.

**Option B (better): a median per group.** Titles in names tell us a lot about age (Master = boy, Miss is often younger). First we need those titles, so we clean text first.

### Fix 3: Text cleaning with `.str`
`.str` gives you string methods for a whole column at once: `.str.strip()`, `.str.title()`, `.str.lower()`, `.str.contains()`, `.str.extract()`.

Names look like `"Braund, Mr. Owen Harris"`. The title sits between the comma and the full stop. `.str.extract` pulls it out with a pattern: `", "` then letters (captured) then `"."`.

In [ ]:
df["Title"] = df["Name"].str.extract(r", ([A-Za-z ]+)\.")
df["Title"] = df["Title"].str.strip()
df["Title"].value_counts()

Many titles are rare. We group them into `Rare`, and treat Mlle/Ms as Miss and Mme as Mrs.

In [ ]:
df["Title"] = df["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
common = ["Mr", "Mrs", "Miss", "Master"]
df.loc[~df["Title"].isin(common), "Title"] = "Rare"
df["Title"].value_counts()

**Justification:** Titles carry age and social information that is buried in the Name. Rare titles have too few rows to learn from, so grouping them keeps results stable.

In [ ]:
# Option B: fill Age with the median age of each title
print(df.groupby("Title")["Age"].median())
df["Age"] = df["Age"].fillna(df.groupby("Title")["Age"].transform("median"))
print("Missing Age after fix:", df["Age"].isnull().sum())

**Justification:** 20% is too many rows to drop. A boy (Master, median about 3.5) should not get an adult age of 28, so the median per title gives more realistic values than one median for everyone. Limitation: filled ages are still estimates.

### Fix 4: Cabin (687 missing, about 77%)
Too much is missing to fill honestly. But "cabin known" might itself be useful, so we keep a True/False flag before dropping.

In [ ]:
df["HasCabin"] = df["Cabin"].notnull()
df = df.drop(columns=["Cabin"])
print(df["HasCabin"].value_counts())

**Justification:** Any filled value would be mostly invented. The flag keeps the one honest piece of information: whether a cabin was recorded.

### `dropna`: whole table vs subset
`dropna()` removes rows with **any** missing value. On the raw data that is dangerous:

In [ ]:
print("Rows if we dropna on everything:", len(raw.dropna()))
print("Rows if we only require Embarked:", len(raw.dropna(subset=["Embarked"])))

Dropping on the whole table would throw away 708 of 891 passengers because of Cabin. Always use `subset=` to say which columns must be present, and print the row count.

### Fix 5: Duplicates

In [ ]:
before = len(df)
df = df.drop_duplicates()
print("Rows removed:", before - len(df))

**Justification:** Duplicates would count the same passenger twice. There were 0, and we printed that so we know for sure.

### Fix 6: Replace codes with readable values
`map` swaps each value using a dictionary. Values not in the dictionary become `NaN`, so check afterwards.

In [ ]:
ports = {"C": "Cherbourg", "Q": "Queenstown", "S": "Southampton"}
df["Embarked"] = df["Embarked"].map(ports)
print(df["Embarked"].value_counts(dropna=False))

**Justification:** Full names are clearer in charts and tables. The meaning is unchanged; the count of missing values is still 0, so no codes were lost.

In [ ]:
# Tidy text: make Sex consistent (already clean here, but a good habit)
df["Sex"] = df["Sex"].str.strip().str.title()
df["Sex"].value_counts()

### Fix 7: Types and names

In [ ]:
df["Age"] = df["Age"].round().astype(int)
df["Pclass"] = df["Pclass"].astype("category")
df = df.rename(columns={"SibSp": "SiblingsSpouses", "Parch": "ParentsChildren"})
df.dtypes

**Justification:** Whole-year ages are easier to read and group (babies become 0). Pclass is a label, not a quantity, so `category` is honest. Renaming does not change values, only makes columns self-explaining.

### Your turn 5: Fare 0
The 15 passengers with `Fare == 0` look odd. Decide what to do (keep, drop, flag, or fill) and write code for it on `df`. Print how many rows `df` has afterwards. Then write your justification below.

In [ ]:
# Your code here


**Justification:** (write yours here)

### Your turn 6: Ticket prefixes
Some tickets start with letters (like `"A/5 21171"`). Create a column `TicketHasLetters` that is True when the ticket contains any letter. Hint: `.str.contains("[A-Za-z]")`. How many are True?

In [ ]:
# Your code here


### Your turn 7: Sex as numbers
Some models need numbers. Use `map` to create `SexCode` with Male = 0 and Female = 1, then check there are no missing values in it.

In [ ]:
# Your code here


## 5. New columns and saving

In [ ]:
df["FamilySize"] = df["SiblingsSpouses"] + df["ParentsChildren"] + 1
df[["Name", "FamilySize"]].head()

A column from a function: `apply` runs your function on each value.

In [ ]:
def fare_band(fare):
    if fare == 0:
        return "Free"
    if fare < 10:
        return "Low"
    if fare < 50:
        return "Medium"
    return "High"

df["FareBand"] = df["Fare"].apply(fare_band)
df["FareBand"].value_counts()

### Your turn 8
Create `IsChild` (True when Age is under 16). How many children are there?

In [ ]:
# Your code here


**Final check and save.** `index=False` stops pandas writing the row numbers as an extra column.

In [ ]:
print("Missing values left:", df.isnull().sum().sum())
print("Raw shape:", raw.shape, " Clean shape:", df.shape)
print("Raw data untouched? Missing Age in raw:", raw["Age"].isnull().sum())
df.to_csv("titanic_clean.csv", index=False)
print("Saved titanic_clean.csv")

### Your turn 9
Read `titanic_clean.csv` back into `check`. Confirm the shape matches `df`, and look at `check.dtypes`. Did `Pclass` stay a category? (Saving to CSV keeps only text, so types must be set again after reading.)

In [ ]:
# Your code here


## 6. First questions (EDA)
Because `Survived` is 0 or 1, its **mean** is the survival rate.

In [ ]:
print("Overall survival rate:", round(df["Survived"].mean() * 100, 1), "%")

In [ ]:
df.groupby("Sex")["Survived"].mean().round(3)

In [ ]:
df.groupby("Pclass", observed=True)["Survived"].mean().round(3)

In [ ]:
# Several numbers at once
df.groupby("Title")["Survived"].agg(["count", "mean"]).sort_values("mean", ascending=False).round(3)

In [ ]:
# Two groups together
df.groupby(["Pclass", "Sex"], observed=True)["Survived"].mean().round(3)

**Interpretation:** Only about 38% survived. Women survived far more (about 74%) than men (about 19%), and first class (about 63%) far more than third (about 24%). First and second class women did best of all. This matches "women and children first", and shows that class, which tracks wealth, mattered a lot.

Careful: these are patterns, not proof of cause. Third class cabins were also further from the lifeboats.

In [ ]:
# value_counts with normalize gives shares
df["FareBand"].value_counts(normalize=True).round(3)

### Your turn 10
1. Survival rate by `Embarked`, sorted from highest to lowest.
2. Survival rate by `FareBand`.
3. Write two sentences interpreting both.

In [ ]:
# Your code here


In [ ]:
# Your code here


**Interpretation:** (write yours here)

### Your turn 11
Did having a recorded cabin (`HasCabin`) go with higher survival? Compute it and say whether you think the cabin itself caused it.

In [ ]:
# Your code here


## 7. Challenge (10 to 15 minutes)
Create `AgeGroup` with `"Child"` for Age under 16 and `"Adult"` otherwise (write a function and use `apply`). Then find the survival rate by `AgeGroup` and `Sex` together. Did children of **both** sexes do better than adults? Be careful, look at each number.

In [ ]:
# Your code here


## Recap
- `pd.read_csv` loads a CSV in one line; `usecols`, `nrows` and `index_col` load only what you need.
- Select with `[]`, `loc` (labels), `iloc` (positions) and boolean filters with `&` and `|`.
- Inspect before fixing: `info`, `describe`, `unique`, `value_counts`, `isnull().sum()`, `duplicated()`.
- Clean on a copy, use `dropna(subset=...)` not a blind `dropna()`, and write a justification for every fix.
- `.str`, `map`, `replace`, `astype`, `rename` tidy columns; `groupby(...)["Survived"].mean()` turns clean data into answers.

## Assignment (covers Week 2 Session 2 and Week 3 Session 1)
**Due:** before our next class. **Submit:** your completed `.ipynb` file through the usual submission form.

**Question: Did travelling alone or with family affect survival?**
1. Start a new notebook with `Titanic-Dataset.csv` in the same folder. Load it and inspect it; list the problems you find.
2. Clean it on a copy. You may make different choices from class, but every fix needs a **Justification:** line. Print row counts before and after any change.
3. Create `FamilySize` (SibSp + Parch + 1) and `Alone` (True when FamilySize is 1).
4. Find the survival rate by `Alone`, and by `Alone` together with `Sex` or `Pclass`.
5. Save your cleaned data with `to_csv`.
6. Finish with **3 findings** in plain words, plus one limitation of your cleaning.

**Rubric (20 points, 0 to 4 each)**
| Area | What earns full marks |
|---|---|
| Correctness | Code runs top to bottom; numbers are right |
| Problem solving | Sensible cleaning choices; question answered |
| Code quality | Clear names, no repeated code, raw data untouched |
| Communication | Justifications and findings are clear and specific |
| Independence | Your own thinking; you can explain every line |

AI tools are allowed, but you must be able to explain every line on our call.